# 01b · Build the class-balanced study set

**Why a second study set.** The first (pilot) study picked images at random among those containing any study class. COCO is dominated by people, so the halves came out as:

| | person | chair | car | bicycle | dog |
|---|---|---|---|---|---|
| A (manual) | 21 | 8 | 1 | 0 | 0 |
| B (assisted) | 20 | 0 | 4 | 2 | 0 |

Dog was never tested, and the hard class (chair) was only in the manual half, so the two modes were compared on different content.

**This set picks images per class:** 4 images for each of person, car, bicycle, dog and chair, each with that class clearly visible (at least 64×64 px), at most 5 objects to label, and, for non-person classes, at most 2 people. Per class, 2 images go to half **A** and 2 to half **B**, dealt by workload so both halves get the same classes *and* similar amounts of work. Images from the pilot study, the model evaluation set and the practice set are all excluded.

In [1]:
import sys, pathlib, importlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "eval" else pathlib.Path.cwd()))
from eval import evallib as E
E = importlib.reload(E)
E.setup()
print("project folder:", E.ROOT)

project folder: C:\Users\yuvra\Desktop\UNI\Ai Vision\AI annotator


In [2]:
coco = E.load_coco()
S = E.load_subsets()
exclude = set(S["model"]) | set(S["effort_a"]) | set(S["effort_b"])
B = E.choose_balanced(coco, exclude)
for cls, ids in B["picked_for"].items():
    print(f"{cls:8} picked {len(ids)} images")

loading annotations into memory...
Done (t=0.60s)
creating index...
index created!
person   picked 4 images
car      picked 4 images
bicycle  picked 4 images
dog      picked 4 images
chair    picked 4 images


In [3]:
import pandas as pd
balance = pd.DataFrame({"A (manual)": E.class_counts(coco, B["effort_a"]),
                        "B (assisted)": E.class_counts(coco, B["effort_b"])}).T
balance["total"] = balance.sum(axis=1)
balance          # objects to label, per class and half: should be similar in both rows

,person,car,bicycle,dog,chair,total
A (manual),8,5,2,2,4,21
B (assisted),4,4,3,3,6,20


In [4]:
from tqdm.auto import tqdm
for i in tqdm(B["effort_a"] + B["effort_b"], desc="images"):
    E.image_path(i)
E.copy_effort_images(B, folder_name="effort_balanced")

c:\Users\yuvra\Desktop\UNI\Ai Vision\AI annotator\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
images:   0%|          | 0/20 [00:00<?, ?it/s]

downloading http://images.cocodataset.org/val2017/000000031620.jpg ...


images:   5%|▌         | 1/20 [00:00<00:18,  1.01it/s]

downloading http://images.cocodataset.org/val2017/000000397681.jpg ...


images:  10%|█         | 2/20 [00:01<00:15,  1.14it/s]

downloading http://images.cocodataset.org/val2017/000000575243.jpg ...


... (32 outputs left out)

images:  95%|█████████▌| 19/20 [00:13<00:00,  1.50it/s]

downloading http://images.cocodataset.org/val2017/000000435003.jpg ...


images: 100%|██████████| 20/20 [00:14<00:00,  1.42it/s]


{'A': WindowsPath('C:/Users/yuvra/Desktop/UNI/Ai Vision/AI annotator/eval/data/effort_balanced/A'),
 'B': WindowsPath('C:/Users/yuvra/Desktop/UNI/Ai Vision/AI annotator/eval/data/effort_balanced/B')}

**Next:** the study, on a fresh data folder: `python -m app.server --data eval/studies/data_study2`. In the class picker tick **person, car, bicycle, dog, chair**. Label `eval/data/effort_balanced/A` in **Manual** (Polygon tool), then `eval/data/effort_balanced/B` in **Assisted**. Then run notebook 03 with `RUN = "balanced"`.